In [ ]:
!pip install PyMuPDF sentence-transformers

In [ ]:
import fitz  # PyMuPDF
import re
from sentence_transformers import SentenceTransformer, util

# --- 1. Load the AI Model ---
# This downloads a pre-trained LLM specifically built for semantic matching
print("🧠 Loading Sentence Transformer Model (This takes a few seconds)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# --- 2. Define the "Alias Ontology" and Resources ---
# This solves the "DSA vs Algorithms" problem.
skill_ontology = {
    "Data Structures and Algorithms": ["dsa", "data structures", "algorithms"],
    "Machine Learning": ["machine learning", "ml", "predictive modeling"],
    "Artificial Intelligence": ["artificial intelligence", "ai"],
    "Natural Language Processing": ["natural language processing", "nlp"],
    "Python": ["python", "python3"],
    "SQL": ["sql", "mysql", "postgresql", "databases"],
    "Amazon Web Services": ["aws", "amazon web services", "cloud"]
}

# A dictionary to suggest resources for missing skills
resource_links = {
    "Data Structures and Algorithms": "https://www.coursera.org/specializations/data-structures-algorithms",
    "Machine Learning": "https://www.coursera.org/learn/machine-learning",
    "Natural Language Processing": "https://www.deeplearning.ai/courses/natural-language-processing-specialization/",
    "Amazon Web Services": "https://aws.amazon.com/training/"
}

# --- 3. Core Functions ---

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + " "
    return text.lower()

def clean_text(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_semantic_match(resume_text, jd_text):
    # Convert texts into semantic vectors
    embeddings1 = model.encode(resume_text, convert_to_tensor=True)
    embeddings2 = model.encode(jd_text, convert_to_tensor=True)

    # Calculate cosine similarity of the meanings
    cosine_scores = util.cos_sim(embeddings1, embeddings2)
    return round(cosine_scores.item() * 100, 2)

def analyze_skills(resume_text, jd_text, ontology):
    missing_skills = []

    # We use regex \b to match exact words (so "ai" doesn't match inside "email")
    for standard_name, aliases in ontology.items():
        # Check if ANY alias of the skill is in the Job Description
        is_in_jd = any(re.search(r'\b' + re.escape(alias) + r'\b', jd_text) for alias in aliases)

        if is_in_jd:
            # If it is in the JD, check if ALL aliases are missing from the Resume
            is_in_resume = any(re.search(r'\b' + re.escape(alias) + r'\b', resume_text) for alias in aliases)
            if not is_in_resume:
                missing_skills.append(standard_name)

    return missing_skills

# --- 4. The Execution Block ---

sample_jd = """
We are looking for a Software Engineer intern. The ideal candidate will have
a strong foundation in Python and DSA.
Familiarity with ML, NLP, and SQL databases is highly preferred.
Bonus points for experience with AWS.
"""

pdf_filename = "Prince Resume .pdf" # <-- UPDATE THIS

try:
    print("⏳ Processing documents...")
    raw_resume = extract_text_from_pdf(pdf_filename)

    cleaned_resume = clean_text(raw_resume)
    cleaned_jd = clean_text(sample_jd.lower())

    print("🧮 Calculating Deep Semantic Match...")
    match_percentage = calculate_semantic_match(cleaned_resume, cleaned_jd)

    print("🔍 Analyzing Skill Gaps using Aliases...")
    missing_skills = analyze_skills(cleaned_resume, cleaned_jd, skill_ontology)

    print("\n" + "="*50)
    print("              📊 DEEP ANALYSIS RESULTS 📊")
    print("="*50)
    print(f"Semantic Match Score: {match_percentage}%\n")

    if missing_skills:
        print("⚠️ Missing Skills & Suggested Resources:")
        for skill in missing_skills:
            print(f"   - {skill}")
            # Check if we have a resource link for this skill
            if skill in resource_links:
                print(f"     🔗 Learn here: {resource_links[skill]}")
    else:
        print("✅ Outstanding! Your resume semantically covers all key requirements.")
    print("="*50)

except Exception as e:
    print(f"\n❌ Error: {e}")

🧠 Loading Sentence Transformer Model (This takes a few seconds)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Processing documents...
🧮 Calculating Deep Semantic Match...
🔍 Analyzing Skill Gaps using Aliases...

              📊 DEEP ANALYSIS RESULTS 📊
Semantic Match Score: 44.19%

⚠️ Missing Skills & Suggested Resources:
   - Data Structures and Algorithms
     🔗 Learn here: https://www.coursera.org/specializations/data-structures-algorithms
   - Natural Language Processing
     🔗 Learn here: https://www.deeplearning.ai/courses/natural-language-processing-specialization/
   - SQL
   - Amazon Web Services
     🔗 Learn here: https://aws.amazon.com/training/


In [ ]:
import re

def clean_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_match_score(resume_text, jd_text):
    # Create a vectorizer that ignores common English stop words
    vectorizer = TfidfVectorizer(stop_words='english')

    # Convert both texts into vectors
    vectors = vectorizer.fit_transform([resume_text, jd_text])

    # Calculate similarity between the first vector (resume) and second (JD)
    similarity = cosine_similarity(vectors[0:1], vectors[1:2])

    # Convert to a percentage
    match_percentage = round(similarity[0][0] * 100, 2)
    return match_percentage

In [ ]:
def find_missing_skills(resume_text, jd_text, skill_database):
    # skill_database is a simple Python list you define, e.g., ['python', 'react', 'machine learning', 'sql']
    missing_skills = []

    for skill in skill_database:
        # If the skill is in the JD but NOT in the Resume
        if skill in jd_text and skill not in resume_text:
            missing_skills.append(skill)

    return missing_skills

In [ ]:
# --- 1. Define a Sample Job Description (JD) ---
# Feel free to paste a real JD from LinkedIn or Internshala here
sample_jd = """
We are looking for a Software Engineer intern. The ideal candidate will have
a strong foundation in Python, Data Structures, and Algorithms.
Familiarity with Machine Learning, NLP, and SQL databases is highly preferred.
Bonus points for experience with cloud platforms like AWS or Docker.
"""

# --- 2. Define a Basic Skill Database ---
# (We will expand this later to cover hundreds of skills, but this is for testing)
tech_skills = ['python', 'java', 'sql', 'machine learning', 'nlp',
               'data structures', 'algorithms', 'react', 'aws', 'docker', 'git']

# --- 3. Execute the Pipeline ---
# IMPORTANT: Change this string to the exact name of the PDF you uploaded to Colab
pdf_filename = "Prince Resume .pdf"

try:
    print("⏳ Extracting text from PDF...")
    raw_resume = extract_text_from_pdf(pdf_filename)

    print("🧹 Cleaning text data...")
    cleaned_resume = clean_text(raw_resume)
    cleaned_jd = clean_text(sample_jd.lower())

    print("🧮 Calculating Match Score...")
    match_percentage = calculate_match_score(cleaned_resume, cleaned_jd)

    print("🔍 Analyzing Skill Gaps...")
    missing_skills = find_missing_skills(cleaned_resume, cleaned_jd, tech_skills)

    # --- 4. Display the Final Output ---
    print("\n" + "="*45)
    print("              📊 ANALYSIS RESULTS 📊")
    print("="*45)
    print(f"Resume Match Score: {match_percentage}%\n")

    if missing_skills:
        print("⚠️ Missing Skills (Found in JD, missing in Resume):")
        for skill in missing_skills:
            print(f"   - {skill.title()}")
    else:
        print("✅ Great job! Your resume covers all the key skills listed.")
    print("="*45)

except Exception as e:
    print(f"\n❌ Error: {e}")
    print("Check that your PDF is uploaded to Colab and the filename matches exactly.")

⏳ Extracting text from PDF...
🧹 Cleaning text data...
🧮 Calculating Match Score...
🔍 Analyzing Skill Gaps...

              📊 ANALYSIS RESULTS 📊
Resume Match Score: 11.07%

⚠️ Missing Skills (Found in JD, missing in Resume):
   - Sql
   - Nlp
   - Data Structures
   - Algorithms
   - Aws
   - Docker
